In [ ]:
import cv2
import os
import numpy as np
from sklearn.model_selection import train_test_split
import joblib
import utils # Import shared utilities

# Path to your dataset
DATA_PATH = r'..\Dataset\training\Cleaned_Training'

# Path to save test models
MODELS_TEST_PATH = r'..\models\models_test'
os.makedirs(MODELS_TEST_PATH, exist_ok=True)
print(f"Models will be saved to: {MODELS_TEST_PATH}")

In [ ]:
# --- EXECUTE LOAD ---
faces, ids, hog_feats, hog_lbls = utils.load_and_augment_data(DATA_PATH)

if len(faces) > 0:
    print(f"✅ Data Loaded successfully.")
    print(f"Total Augmented Samples: {len(faces)}")
else:
    print("⚠️ No data found. Check your path.")

In [ ]:
# --- DATA SPLITTING ---
# Splitting HOG features (SVM/XGBoost) - 75% Train, 25% Test
X_train_hog, X_test_hog, y_train_hog, y_test_hog = train_test_split(
    hog_feats, hog_lbls, test_size=0.25, random_state=42, stratify=hog_lbls
)

# Splitting Raw Faces (LBPH) - 75% Train, 25% Test
faces_train, faces_test, ids_train, ids_test = train_test_split(
    faces, ids, test_size=0.25, random_state=42, stratify=ids
)

print(f"Training Set Size: {len(faces_train)}")
print(f"Testing Set Size: {len(faces_test)}")

In [ ]:
# --- LBPH EVALUATION ---
if len(faces_train) > 0:
    # Train using UTILS function
    save_path = os.path.join(MODELS_TEST_PATH, 'trainer_test.yml')
    lbph = utils.train_lbph(faces_train, ids_train, save_path=save_path)
    
    print("Evaluating LBPH...")
    y_pred_lbph = []
    for face in faces_test:
        label, confidence = lbph.predict(face)
        y_pred_lbph.append(label)
        
    utils.print_metrics("LBPH", ids_test, y_pred_lbph)

In [ ]:
# --- SVM EVALUATION ---
if len(X_train_hog) > 0:
    # Train using UTILS function
    save_path = os.path.join(MODELS_TEST_PATH, 'svm_face_model_test.pkl')
    svm = utils.train_svm(X_train_hog, y_train_hog, save_path=save_path)
    
    print("Evaluating SVM...")
    y_pred_svm = svm.predict(X_test_hog)
    utils.print_metrics("SVM", y_test_hog, y_pred_svm)

In [ ]:
# --- XGBoost EVALUATION ---
if len(X_train_hog) > 0:
    # Train using UTILS function
    save_path_xgb = os.path.join(MODELS_TEST_PATH, 'xgb_face_model_test.pkl')
    save_path_le = os.path.join(MODELS_TEST_PATH, 'label_encoder_test.pkl')
    
    xgb_model, le = utils.train_xgboost(X_train_hog, y_train_hog, 
                                        save_path_model=save_path_xgb, 
                                        save_path_le=save_path_le)
    
    print("Evaluating XGBoost...")
    # Note: XGBoost prediction needs encoded labels if we use predict directly from the wrapper?
    # No, our training wrapper handles training with encoding.
    # But for prediction here, we need to handle it or use the model's predict which returns indices.
    # The utils.train_xgboost returns (xgb_model, le).
    
    y_pred_xgb_enc = xgb_model.predict(X_test_hog)
    y_pred_xgb = le.inverse_transform(y_pred_xgb_enc)
    
    utils.print_metrics("XGBoost", y_test_hog, y_pred_xgb)

In [ ]:
# --- STACKER EVALUATION ---

if 'lbph' in locals() and 'svm' in locals() and 'xgb_model' in locals():
    print("Initializing Stacker with Test Models...")
    
    # Create a dummy ID map (not strictly needed for metrics, but required by init)
    id_map = {uid: f"User {uid}" for uid in np.unique(hog_lbls)}
    
    # Initialize Stacker
    models = {
        'LBPH': lbph,
        'SVM': svm,
        'XGB': xgb_model,
        'LE': le
    }
    stacker = utils.StackedFaceRecognizer(models, X_train_hog, y_train_hog, id_map)
    
    print("Evaluating Stacker on Test Set...")
    y_true_stacker = ids_test
    y_pred_stacker = []
    unclassified_count = 0
    
    # Simulate Frame-by-Frame Prediction for Test Set
    for idx, face_img in enumerate(faces_test):
        hog_vec = X_test_hog[idx]
        
        # 1. Get Individual Predictions
        preds = {}
        
        # SVM
        try:
            prob_svm = svm.predict_proba([hog_vec])[0]
            conf_svm = np.max(prob_svm)
            pid_svm = svm.classes_[np.argmax(prob_svm)]
            preds['SVM'] = {'id': pid_svm, 'conf': conf_svm}
        except: pass
        
        # XGB
        try:
            prob_xgb = xgb_model.predict_proba([hog_vec])[0]
            conf_xgb = np.max(prob_xgb)
            decoded_id = le.inverse_transform([np.argmax(prob_xgb)])[0]
            preds['XGB'] = {'id': decoded_id, 'conf': conf_xgb}
        except: pass
        
        # LBPH
        try:
            pid_lbph, dist_lbph = lbph.predict(face_img)
            preds['LBPH'] = {'id': pid_lbph, 'conf': dist_lbph}
        except: pass
        
        # 2. Stacker Prediction
        stacker._reset_session() # IMPORTANT: Reset memory for independent test samples
        final_id, final_conf, method = stacker._predict_single(face_img, preds)
        
        if final_id is None:
            # Handle unclassified cases (assign a dummy ID that will count as error)
            # We use 99999 as it's likely not a valid ID
            final_id = 99999
            unclassified_count += 1
            
        y_pred_stacker.append(final_id)

    print(f"Stacker Evaluation Complete. Unclassified instances: {unclassified_count}")
    utils.print_metrics("Stacker", y_true_stacker, y_pred_stacker)
    
    # Save Stacker (The class object)
    # NOTE: OpenCV's LBPHFaceRecognizer cannot be pickled. We remove it before saving.
    # It is saved separately as 'trainer_test.yml'.
    stacker.lbph = None 

    stacker_path = os.path.join(MODELS_TEST_PATH, 'stacker_model_test.pkl')
    joblib.dump(stacker, stacker_path)
    print(f"✅ Saved Stacker Object (excluding LBPH) to '{stacker_path}'")
